In [1]:
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, TensorDataset
from transformers import AutoModel
from sklearn.metrics import f1_score
import pandas as pd
import matplotlib.pyplot as plt
import subprocess

# Download data
subprocess.run(["kaggle", "datasets", "download",
                "-d", "kaggler10240/msr-data", "--unzip"], check=True)

import os
os.makedirs("/root/.config/kaggle", exist_ok=True)
with open("/root/.config/kaggle/kaggle.json", "w") as f:
    f.write('{"username":"YOUR_USERNAME","key":"YOUR_KEY"}')
os.chmod("/root/.config/kaggle/kaggle.json", 0o600)

# Load preprocessed data — copy from main notebook output
print("Loading data...")
train_data = torch.load('train_data.pt')
val_data   = torch.load('val_data.pt')

train_dataset = TensorDataset(
    train_data['input_ids'],
    train_data['attention_mask'],
    train_data['labels']
)
val_dataset = TensorDataset(
    val_data['input_ids'],
    val_data['attention_mask'],
    val_data['labels']
)

train_loader = DataLoader(train_dataset, batch_size=16, shuffle=True)
val_loader   = DataLoader(val_dataset,   batch_size=16, shuffle=False)

# Build model
class SecureScanModel(nn.Module):
    def __init__(self):
        super(SecureScanModel, self).__init__()
        self.codebert = AutoModel.from_pretrained("microsoft/codebert-base")
        for i in range(6):
            for param in self.codebert.encoder.layer[i].parameters():
                param.requires_grad = False
        self.lstm = nn.LSTM(768, 256, num_layers=2, batch_first=True,
                            bidirectional=True, dropout=0.3)
        self.classifier = nn.Sequential(
            nn.Linear(512, 256), nn.ReLU(), nn.BatchNorm1d(256),
            nn.Dropout(0.3), nn.Linear(256, 128), nn.ReLU(),
            nn.Dropout(0.3), nn.Linear(128, 2)
        )
    def forward(self, input_ids, attention_mask):
        out = self.codebert(input_ids=input_ids, attention_mask=attention_mask)
        lstm_out, _ = self.lstm(out.last_hidden_state)
        return self.classifier(lstm_out[:, 0, :])

def train_one_epoch(model, loader, optimizer, criterion):
    model.train()
    preds, labels_list = [], []
    for input_ids, mask, labels in loader:
        input_ids, mask, labels = input_ids.cuda(), mask.cuda(), labels.cuda()
        optimizer.zero_grad()
        loss = criterion(model(input_ids, mask), labels)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        preds.extend(torch.argmax(model(input_ids, mask), 1).cpu().numpy())
        labels_list.extend(labels.cpu().numpy())
    return f1_score(labels_list, preds, average='macro')

def val_one_epoch(model, loader, criterion):
    model.eval()
    preds, labels_list = [], []
    with torch.no_grad():
        for input_ids, mask, labels in loader:
            input_ids, mask, labels = input_ids.cuda(), mask.cuda(), labels.cuda()
            preds.extend(torch.argmax(model(input_ids, mask), 1).cpu().numpy())
            labels_list.extend(labels.cpu().numpy())
    return f1_score(labels_list, preds, average='macro')

criterion = nn.CrossEntropyLoss(weight=torch.tensor([1.0, 3.0]).cuda())
results = {}

for optimizer_name in ['Adam', 'SGD']:
    print(f"\nTraining with {optimizer_name}...")
    model = SecureScanModel().cuda()
    
    if optimizer_name == 'Adam':
        optimizer = torch.optim.Adam(
            filter(lambda p: p.requires_grad, model.parameters()), lr=2e-5)
    else:
        optimizer = torch.optim.SGD(
            filter(lambda p: p.requires_grad, model.parameters()),
            lr=0.01, momentum=0.9)
    
    history = []
    for epoch in range(3):
        train_f1 = train_one_epoch(model, train_loader, optimizer, criterion)
        val_f1   = val_one_epoch(model, val_loader, criterion)
        history.append({'epoch': epoch+1, 'train_f1': train_f1, 'val_f1': val_f1})
        print(f"  Epoch {epoch+1}/3 — Train F1: {train_f1:.4f} | Val F1: {val_f1:.4f}")
    
    results[optimizer_name] = history
    print(f"{optimizer_name} best Val F1: {max(h['val_f1'] for h in history):.4f}")

# Plot comparison
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for i, opt in enumerate(['Adam', 'SGD']):
    epochs = [h['epoch'] for h in results[opt]]
    val_f1s = [h['val_f1'] for h in results[opt]]
    axes[0].plot(epochs, val_f1s, label=opt, marker='o')

axes[0].set_title('Adam vs SGD — Val F1')
axes[0].set_xlabel('Epoch')
axes[0].set_ylabel('F1 Score')
axes[0].legend()
axes[0].grid(True)

# Bar chart of best F1
best_f1s = {opt: max(h['val_f1'] for h in results[opt]) for opt in results}
axes[1].bar(best_f1s.keys(), best_f1s.values(), color=['blue', 'orange'])
axes[1].set_title('Best Val F1 Comparison')
axes[1].set_ylabel('F1 Score')
axes[1].set_ylim(0.5, 1.0)
for i, (opt, val) in enumerate(best_f1s.items()):
    axes[1].text(i, val + 0.01, f'{val:.4f}', ha='center')
axes[1].grid(True, axis='y')

plt.tight_layout()
plt.savefig('optimizer_comparison.png', dpi=150)
plt.show()
print("\nOptimizer comparison done and saved!")

Dataset URL: https://www.kaggle.com/datasets/kaggler10240/msr-data
License(s): unknown


100%|██████████| 1.47G/1.47G [00:10<00:00, 156MB/s] 



Loading data...


FileNotFoundError: [Errno 2] No such file or directory: 'train_data.pt'